# 🧠 Brain Tumor Detection using Deep Learning

This notebook provides an end-to-end pipeline for classifying brain MRI scans into **4 categories**:
- **Glioma** — Tumor originating from glial cells
- **Meningioma** — Tumor arising from the meninges
- **Pituitary** — Tumor in the pituitary gland
- **No Tumor** — Healthy brain scan

We train and compare **3 models**:
1. Custom CNN (built from scratch)
2. VGG16 (transfer learning)
3. ResNet50 (transfer learning)

---

## 1. Setup & Configuration

In [ ]:
# ── Google Colab Setup ──────────────────────────────────────────────
import os, sys

# Detect if running on Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('🔄 Running on Google Colab — setting up environment...')
    # Mount Google Drive (optional — for saving models)
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Install any missing packages
    !pip install -q tensorflow scikit-learn matplotlib seaborn opencv-python Pillow tqdm
    
    # Check GPU availability
    !nvidia-smi
else:
    print('🖥️ Running locally')

# Check TensorFlow GPU
import tensorflow as tf
print(f'\nTensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
import warnings
from pathlib import Path
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
sns.set_theme(style='darkgrid')

print('✅ All imports successful!')

In [ ]:
# ── Constants ────────────────────────────────────────────────────────
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_LABELS = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES = len(CLASS_LABELS)
SEED = 42

# Set paths — adjust these to your dataset location
if IN_COLAB:
    BASE_DIR = '/content/brain_tumor_dataset'
else:
    BASE_DIR = str(Path('..').resolve())  # project root

TRAIN_DIR = os.path.join(BASE_DIR, 'dataset', 'Training')
TEST_DIR = os.path.join(BASE_DIR, 'dataset', 'Testing')
MODEL_DIR = os.path.join(BASE_DIR, 'models')
EVAL_DIR = os.path.join(BASE_DIR, 'evaluation', 'plots')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

print(f'Train directory: {TRAIN_DIR}')
print(f'Test directory:  {TEST_DIR}')
print(f'Model directory: {MODEL_DIR}')

## 2. Dataset Download (Colab Only)

If running on Colab, upload your `kaggle.json` API key and download the dataset.

In [ ]:
if IN_COLAB:
    # Upload kaggle.json
    from google.colab import files
    print('Upload your kaggle.json file:')
    uploaded = files.upload()
    
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    
    # Download dataset
    !pip install -q kaggle
    !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p /content/
    !unzip -q /content/brain-tumor-mri-dataset.zip -d /content/brain_tumor_dataset/
    
    print('\n✅ Dataset downloaded and extracted!')
else:
    print('Running locally — ensure dataset is in the dataset/ folder.')
    print('Run: python dataset/download_dataset.py')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Count images per class ───────────────────────────────────────────
def count_images(directory):
    """Count images in each class subdirectory."""
    counts = {}
    if not os.path.exists(directory):
        print(f'⚠️ Directory not found: {directory}')
        return counts
    for class_name in sorted(os.listdir(directory)):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            count = len([f for f in os.listdir(class_path) 
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            counts[class_name] = count
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)

print('📊 Dataset Distribution:')
print(f'\nTraining Set ({sum(train_counts.values())} images):')
for cls, cnt in train_counts.items():
    print(f'  {cls:15s}: {cnt:5d} images')

print(f'\nTesting Set ({sum(test_counts.values())} images):')
for cls, cnt in test_counts.items():
    print(f'  {cls:15s}: {cnt:5d} images')

In [ ]:
# ── Class Distribution Bar Chart ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#ff6b6b', '#ffd93d', '#6bcb77', '#4d96ff']

# Training
axes[0].bar(train_counts.keys(), train_counts.values(), color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Training Set Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Images')
for i, (k, v) in enumerate(train_counts.items()):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Testing
axes[1].bar(test_counts.keys(), test_counts.values(), color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_title('Testing Set Distribution', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Number of Images')
for i, (k, v) in enumerate(test_counts.items()):
    axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: class_distribution.png')

In [ ]:
# ── Sample Images Grid ──────────────────────────────────────────────
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
fig.suptitle('Sample MRI Images by Class', fontsize=16, fontweight='bold', y=1.02)

for row_idx, class_name in enumerate(sorted(os.listdir(TRAIN_DIR))):
    class_path = os.path.join(TRAIN_DIR, class_name)
    if not os.path.isdir(class_path):
        continue
    images = sorted(os.listdir(class_path))[:5]
    for col_idx, img_name in enumerate(images):
        img_path = os.path.join(class_path, img_name)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[row_idx][col_idx].imshow(img)
        axes[row_idx][col_idx].axis('off')
        if col_idx == 0:
            axes[row_idx][col_idx].set_ylabel(
                class_name.upper(), fontsize=11, fontweight='bold', rotation=0, 
                labelpad=70, va='center'
            )

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: sample_images.png')

## 4. Data Preprocessing & Augmentation

In [ ]:
# ── Data Generators ─────────────────────────────────────────────────

# Training generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=0.2,  # 20% for validation
)

# Testing generator — only rescale
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

# Create generators
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED,
    shuffle=True,
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED,
    shuffle=False,
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)

print(f'\n📋 Class indices: {train_generator.class_indices}')
print(f'Training samples:   {train_generator.samples}')
print(f'Validation samples: {val_generator.samples}')
print(f'Testing samples:    {test_generator.samples}')

In [ ]:
# ── Visualize Augmented Images ──────────────────────────────────────
sample_batch, _ = next(train_generator)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Augmented Training Images', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(sample_batch):
        ax.imshow(sample_batch[i])
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'augmented_samples.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Data augmentation working correctly')

## 5. Model Architectures

### 5.1 Custom CNN

In [ ]:
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=4):
    """Build a custom 4-block CNN from scratch."""
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Block 2
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Block 3
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Block 4
        layers.Conv2D(256, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Classification Head
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax'),
    ])
    return model

custom_cnn = build_custom_cnn()
custom_cnn.summary()

### 5.2 VGG16 Transfer Learning

In [ ]:
def build_vgg16_model(input_shape=(224, 224, 3), num_classes=4):
    """Build VGG16 transfer learning model."""
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    
    # Freeze base layers
    for layer in base_model.layers:
        layer.trainable = False
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ])
    return model

vgg16_model = build_vgg16_model()
print(f'VGG16 Total params: {vgg16_model.count_params():,}')
print(f'Trainable params: {sum(tf.keras.backend.count_params(w) for w in vgg16_model.trainable_weights):,}')

### 5.3 ResNet50 Transfer Learning

In [ ]:
def build_resnet50_model(input_shape=(224, 224, 3), num_classes=4):
    """Build ResNet50 transfer learning model."""
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    
    # Freeze base layers
    for layer in base_model.layers:
        layer.trainable = False
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ])
    return model

resnet50_model = build_resnet50_model()
print(f'ResNet50 Total params: {resnet50_model.count_params():,}')
print(f'Trainable params: {sum(tf.keras.backend.count_params(w) for w in resnet50_model.trainable_weights):,}')

## 6. Training

### Common Callbacks & Training Function

In [ ]:
def get_callbacks(model_name):
    """Get training callbacks."""
    return [
        callbacks.EarlyStopping(
            monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1
        ),
        callbacks.ModelCheckpoint(
            os.path.join(MODEL_DIR, f'{model_name}.h5'),
            monitor='val_accuracy', save_best_only=True, verbose=1
        ),
    ]

def train_model(model, model_name, epochs=50, lr=0.001):
    """Compile and train a model."""
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    
    print(f'\n{"="*60}')
    print(f'🏋️ Training {model_name}')
    print(f'{"="*60}')
    
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=epochs,
        callbacks=get_callbacks(model_name),
        verbose=1,
    )
    
    return history

print('✅ Training functions defined')

### 6.1 Train Custom CNN

In [ ]:
custom_cnn = build_custom_cnn()
history_custom = train_model(custom_cnn, 'custom_cnn', epochs=50, lr=0.001)

### 6.2 Train VGG16 (2-Phase: Frozen → Fine-tune)

In [ ]:
# Phase 1: Train only the new classification head
vgg16_model = build_vgg16_model()
history_vgg_phase1 = train_model(vgg16_model, 'vgg16_transfer', epochs=10, lr=0.001)

# Phase 2: Unfreeze last 4 VGG layers and fine-tune
print('\n🔓 Unfreezing last 4 VGG16 layers for fine-tuning...')
base = vgg16_model.layers[0]  # VGG16 base
for layer in base.layers[-4:]:
    layer.trainable = True

history_vgg_phase2 = train_model(vgg16_model, 'vgg16_transfer', epochs=20, lr=1e-5)

### 6.3 Train ResNet50 (2-Phase: Frozen → Fine-tune)

In [ ]:
# Phase 1: Train only the new classification head
resnet50_model = build_resnet50_model()
history_resnet_phase1 = train_model(resnet50_model, 'resnet50_transfer', epochs=10, lr=0.001)

# Phase 2: Unfreeze last 10 ResNet layers and fine-tune
print('\n🔓 Unfreezing last 10 ResNet50 layers for fine-tuning...')
base = resnet50_model.layers[0]  # ResNet50 base
for layer in base.layers[-10:]:
    layer.trainable = True

history_resnet_phase2 = train_model(resnet50_model, 'resnet50_transfer', epochs=20, lr=1e-5)

## 7. Training History Visualization

In [ ]:
def plot_training_history(history, model_name):
    """Plot training & validation loss and accuracy."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name} — Training History', fontsize=14, fontweight='bold')
    
    # Loss
    axes[0].plot(history.history['loss'], label='Train Loss', color='#ff6b6b', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Val Loss', color='#4d96ff', linewidth=2)
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history.history['accuracy'], label='Train Acc', color='#6bcb77', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], label='Val Acc', color='#ffd93d', linewidth=2)
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(EVAL_DIR, f'{model_name}_history.png'), dpi=150, bbox_inches='tight')
    plt.show()

# Plot for all models
plot_training_history(history_custom, 'Custom CNN')
plot_training_history(history_vgg_phase2, 'VGG16 Transfer')
plot_training_history(history_resnet_phase2, 'ResNet50 Transfer')

## 8. Model Evaluation & Comparison

In [ ]:
def evaluate_model(model, model_name, generator):
    """Evaluate a model and generate comprehensive metrics."""
    print(f'\n{"="*60}')
    print(f'📊 Evaluating: {model_name}')
    print(f'{"="*60}')
    
    # Test loss and accuracy
    loss, accuracy = model.evaluate(generator, verbose=0)
    print(f'Test Loss:     {loss:.4f}')
    print(f'Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')
    
    # Predictions
    predictions = model.predict(generator, verbose=0)
    y_pred = np.argmax(predictions, axis=1)
    y_true = generator.classes
    class_labels = list(generator.class_indices.keys())
    
    # Classification Report
    print(f'\nClassification Report:')
    report = classification_report(y_true, y_pred, target_names=class_labels)
    print(report)
    
    return {
        'name': model_name,
        'loss': loss,
        'accuracy': accuracy,
        'y_pred': y_pred,
        'y_true': y_true,
        'predictions': predictions,
        'class_labels': class_labels,
    }

# Evaluate all models
results_custom = evaluate_model(custom_cnn, 'Custom CNN', test_generator)
results_vgg = evaluate_model(vgg16_model, 'VGG16 Transfer', test_generator)
results_resnet = evaluate_model(resnet50_model, 'ResNet50 Transfer', test_generator)

In [ ]:
# ── Confusion Matrices ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Confusion Matrices', fontsize=16, fontweight='bold')

for ax, result in zip(axes, [results_custom, results_vgg, results_resnet]):
    cm = confusion_matrix(result['y_true'], result['y_pred'])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=result['class_labels'],
        yticklabels=result['class_labels'],
    )
    ax.set_title(f"{result['name']}\nAcc: {result['accuracy']:.2%}", fontsize=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC Curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('ROC Curves (One-vs-All)', fontsize=16, fontweight='bold')

colors_roc = ['#ff6b6b', '#ffd93d', '#6bcb77', '#4d96ff']

for ax, result in zip(axes, [results_custom, results_vgg, results_resnet]):
    y_true_bin = label_binarize(result['y_true'], classes=range(NUM_CLASSES))
    
    for i, class_name in enumerate(result['class_labels']):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], result['predictions'][:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors_roc[i], linewidth=2,
                label=f'{class_name} (AUC={roc_auc:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_title(result['name'], fontsize=12)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Model Comparison Table ──────────────────────────────────────────
comparison = pd.DataFrame({
    'Model': [r['name'] for r in [results_custom, results_vgg, results_resnet]],
    'Test Accuracy': [f"{r['accuracy']:.2%}" for r in [results_custom, results_vgg, results_resnet]],
    'Test Loss': [f"{r['loss']:.4f}" for r in [results_custom, results_vgg, results_resnet]],
})

print('\n📊 Model Comparison:')
print(comparison.to_string(index=False))

# Find the best model
best_idx = np.argmax([r['accuracy'] for r in [results_custom, results_vgg, results_resnet]])
best_name = [results_custom, results_vgg, results_resnet][best_idx]['name']
print(f'\n🏆 Best Model: {best_name}')

## 9. Grad-CAM Visualization

Grad-CAM highlights which regions of the MRI the model focuses on for its prediction.

In [ ]:
def generate_gradcam(model, img_array, class_idx, last_conv_layer_name=None):
    """Generate Grad-CAM heatmap for a given image and class."""
    # Auto-detect last conv layer if not specified
    if last_conv_layer_name is None:
        for layer in reversed(model.layers):
            if isinstance(layer, tf.keras.layers.Conv2D):
                last_conv_layer_name = layer.name
                break
            if hasattr(layer, 'layers'):
                for sub in reversed(layer.layers):
                    if isinstance(sub, tf.keras.layers.Conv2D):
                        last_conv_layer_name = sub.name
                        break
                if last_conv_layer_name:
                    break
    
    # Build gradient model
    # Handle nested models (transfer learning)
    try:
        last_conv_layer = model.get_layer(last_conv_layer_name)
    except ValueError:
        # Try looking inside nested models
        for layer in model.layers:
            if hasattr(layer, 'get_layer'):
                try:
                    last_conv_layer = layer.get_layer(last_conv_layer_name)
                    grad_model = tf.keras.models.Model(
                        inputs=model.input,
                        outputs=[layer.get_layer(last_conv_layer_name).output, model.output]
                    )
                    break
                except ValueError:
                    continue
    else:
        grad_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=[last_conv_layer.output, model.output]
        )
    
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, class_idx]
    
    grads = tape.gradient(loss, conv_outputs)
    if grads is None:
        return np.zeros(IMAGE_SIZE)
    
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    
    return heatmap.numpy()

print('✅ Grad-CAM function defined')

In [ ]:
# ── Visualize Grad-CAM for sample test images ──────────────────────
# Get a batch of test images
test_generator.reset()
test_images, test_labels = next(test_generator)

# Use the best model for Grad-CAM
best_model = [custom_cnn, vgg16_model, resnet50_model][best_idx]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle(f'Grad-CAM Visualization ({best_name})', fontsize=16, fontweight='bold')

for i in range(3):
    img = test_images[i]
    img_array = np.expand_dims(img, axis=0)
    
    pred = best_model.predict(img_array, verbose=0)
    pred_class = np.argmax(pred)
    true_class = np.argmax(test_labels[i])
    
    heatmap = generate_gradcam(best_model, img_array, pred_class)
    heatmap_resized = cv2.resize(heatmap, IMAGE_SIZE)
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB) / 255.0
    overlay = 0.6 * img + 0.4 * heatmap_colored
    
    class_labels_map = list(test_generator.class_indices.keys())
    
    # Original
    axes[i][0].imshow(img)
    axes[i][0].set_title('Original MRI')
    axes[i][0].axis('off')
    
    # Heatmap
    axes[i][1].imshow(heatmap_resized, cmap='jet')
    axes[i][1].set_title('Grad-CAM Heatmap')
    axes[i][1].axis('off')
    
    # Overlay
    axes[i][2].imshow(np.clip(overlay, 0, 1))
    axes[i][2].set_title('Overlay')
    axes[i][2].axis('off')
    
    # Prediction info
    axes[i][3].axis('off')
    info_text = (
        f"True: {class_labels_map[true_class]}\n"
        f"Pred: {class_labels_map[pred_class]}\n"
        f"Conf: {pred[0][pred_class]:.2%}\n"
        f"{'✅ Correct' if pred_class == true_class else '❌ Wrong'}"
    )
    axes[i][3].text(0.5, 0.5, info_text, transform=axes[i][3].transAxes,
                    fontsize=12, verticalalignment='center', horizontalalignment='center',
                    fontfamily='monospace',
                    bbox=dict(boxstyle='round', facecolor='#112240', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'gradcam_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: gradcam_visualization.png')

## 10. Conclusion

### Summary

In [ ]:
print('='*60)
print('🧠 BRAIN TUMOR DETECTION — FINAL RESULTS')
print('='*60)
print()
print(comparison.to_string(index=False))
print()
print(f'🏆 Best Model: {best_name}')
print()
print('📁 Saved Files:')
print(f'  Models:  {MODEL_DIR}/')
print(f'  Plots:   {EVAL_DIR}/')
print()
print('🚀 Next Steps:')
print('  1. Start the FastAPI backend:  uvicorn app.backend.main:app --reload')
print('  2. Start the React frontend:   cd app/frontend && npm run dev')
print('  3. Upload MRI images and test predictions!')
print()
print('⚠️  Disclaimer: This tool is for educational purposes only.')
print('    Always consult medical professionals for diagnosis.')